# Расчёты и проверки для устава

Все числа в `docs/project_charter.md` получены здесь. Порядок работы один:

1. тетрадь считает показатели и записывает их в сводку (функция `record`);
2. последняя ячейка печатает сводку — раздел устава, показатель, значение;
3. устав берёт числа из сводки.

Утверждения устава вида «ключ уникален» или «доля оттока растёт» проверяются через `assert`: если утверждение ложно, ячейка падает и тетрадь останавливается. Чисел, вписанных вручную, в проверках нет.

Тетрадь только читает исходные CSV и ничего не записывает. Номера разделов совпадают с разделами устава.

### Как запустить в Google Colab

1. Создай в корне Google Drive («Мой диск») папку `tsu_practicum` и загрузи в неё:
   - `subscribers.csv`, `billing_monthly.csv`, `contract_events.csv`, `billing_preview.csv`, `data_dictionary.csv`;
   - папку `src` с файлом `sources.py` внутри.
2. Открой эту тетрадь в Drive: правый клик → «Открыть с помощью» → «Google Colaboratory».
3. Меню «Среда выполнения» → «Выполнить все». Colab попросит доступ к Drive — разреши.

Если папка лежит в другом месте, поменяй `DRIVE_FOLDER` в первой ячейке.

Локально тетрадь работает без изменений: запускать из папки `notebooks/` сверху вниз.

## Подготовка

In [1]:
import sys
from pathlib import Path

import pandas as pd

DRIVE_FOLDER = "tsu_practicum"  # папка в «Мой диск»

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

required = ["subscribers.csv", "billing_monthly.csv", "contract_events.csv",
            "billing_preview.csv", "data_dictionary.csv", "src/sources.py"]
missing = [f for f in required if not (ROOT / f).exists()]
if missing:
    raise FileNotFoundError(f"В {ROOT} не найдены: {missing}")
print("Данные:", ROOT, "| pandas", pd.__version__)

sys.path.insert(0, str(ROOT / "src"))
from sources import read, to_num, parse_billing_month, contract_dates, active_base, snapshot_table

pd.set_option("display.width", 200)

Данные: /home/claude/tsu_practicum | pandas 3.0.5


In [2]:
figures = []

def record(section, name, value):
    """Записать показатель в сводку для устава и вернуть его."""
    figures.append({"раздел": section, "показатель": name, "значение": value})
    return value

In [3]:
subs = read("subscribers.csv")
events = read("contract_events.csv")
billing_raw = read("billing_monthly.csv")
preview = read("billing_preview.csv")
dictionary = read("data_dictionary.csv")

billing_raw["bm"] = parse_billing_month(billing_raw.billing_month)

# Даты договора — только из журнала. contract_dates падает, если у абонента
# не одно подключение или больше одного расторжения.
act, term = contract_dates(events)

# Биллинг без полных дублей: одна строка на (customer_id, месяц)
billing = billing_raw.drop(columns="billing_month").drop_duplicates().copy()
billing["month_end"] = billing.bm + pd.DateOffset(months=1)
billing["available"] = pd.to_datetime(billing.available_at)
billing["act"] = billing.customer_id.map(act)
billing["term"] = billing.customer_id.map(term)
never = pd.Timestamp("2100-01-01")

## 3. Источники и единицы наблюдения

In [4]:
for name, df in [("subscribers.csv", subs), ("billing_monthly.csv", billing_raw),
                 ("contract_events.csv", events), ("billing_preview.csv", preview),
                 ("data_dictionary.csv", dictionary)]:
    record("3", f"строк в {name}", len(df))
pd.DataFrame(figures)

,раздел,показатель,значение
0,3,строк в subscribers.csv,2600
1,3,строк в billing_monthly.csv,24248
2,3,строк в contract_events.csv,3299
3,3,строк в billing_preview.csv,60
4,3,строк в data_dictionary.csv,23


In [5]:
# customer_id уникален; абоненты справочника и журнала совпадают; event_id уникален
assert not subs.customer_id.duplicated().any()
assert set(subs.customer_id) == set(events.customer_id)
assert not events.event_id.duplicated().any()

record("3", "подключений", int((events.event_type == "activation").sum()))
record("3", "расторжений", int((events.event_type == "termination").sum()))

699

In [6]:
# activation_date справочника совпадает с журналом там, где заполнена
filled = subs[subs.activation_date != ""].set_index("customer_id").activation_date
assert (pd.to_datetime(filled) == act[filled.index]).all()

record("3", "activation_date заполнена", len(filled))
record("3", "activation_date пуста", int((subs.activation_date == "").sum()))

65

In [7]:
# Все строки billing_preview есть в основном биллинге
assert len(preview.merge(billing_raw[preview.columns.tolist()], how="inner")) == len(preview)

### Покрытие биллинга

Строка биллинга должна быть ровно в тех месяцах, где у договора был хотя бы один активный день.

In [8]:
months = pd.date_range(billing.bm.min(), billing.bm.max(), freq="MS")
expected = {
    (c, m)
    for c, a in act.items()
    for m in months
    if a < m + pd.DateOffset(months=1) and (term[c] if pd.notna(term[c]) else never) > m
}
actual = set(zip(billing.customer_id, billing.bm))
assert len(actual) == len(billing), "после удаления полных дублей ключ (customer_id, месяц) уникален"
assert expected == actual

record("3", "строк биллинга после удаления дублей = ожидаемых", len(actual))
record("3", "месячных строк на абонента, максимум", int(billing.groupby("customer_id").size().max()))
record("3", "событий на абонента, максимум", int(events.groupby("customer_id").size().max()))

2

In [9]:
# active_days = число дней месяца внутри [подключение, расторжение)
start = billing[["bm", "act"]].max(axis=1)
end = pd.concat([billing.month_end, billing.term.fillna(never)], axis=1).min(axis=1)
assert ((end - start).dt.days == billing.active_days.astype(int)).all()

# available_at = 00:00 первого дня следующего месяца
assert (billing.available == billing.month_end).all()

In [10]:
# Долг: closing = opening + charged - paid; opening = closing прошлого месяца; первая строка без долга
opening, charged, paid, closing = (to_num(billing[c]) for c in
    ["opening_debt_rub", "charged_rub", "paid_rub", "closing_debt_rub"])
has_closing = closing.notna()
assert ((opening + charged - paid - closing).abs()[has_closing] < 0.01).all()

ordered = billing.assign(opening=opening, closing=closing).sort_values(["customer_id", "bm"])
prev_closing = ordered.groupby("customer_id").closing.shift()
assert ((ordered.opening - prev_closing).abs()[prev_closing.notna()] < 0.01).all()
assert (ordered.groupby("customer_id").opening.first() == 0).all()

In [11]:
# Абоненты без биллинга подключились в сентябре 2026: их месяц ещё не выгружен
no_billing = act.index.difference(billing.customer_id.unique())
assert (act[no_billing] >= "2026-09-01").all()

record("3", "абонентов без биллинга", len(no_billing))
record("3", "их подключения, с", str(act[no_billing].min().date()))
record("3", "их подключения, по", str(act[no_billing].max().date()))

'2026-09-29'

### Дефекты биллинга

In [12]:
full_dups = billing_raw.drop(columns="billing_month").duplicated()
key_dups = billing_raw.duplicated(["customer_id", "bm"])
assert (full_dups == key_dups).all(), "все дубли ключа — полные дубли строк"

record("3", "дефект: полные дубли строк", int(full_dups.sum()))
record("3", "дефект: billing_month в формате ДД.ММ.ГГГГ",
       int(billing_raw.billing_month.str.match(r"^\d{2}\.\d{2}\.\d{4}$").sum()))
record("3", "дефект: нестандартное написание tariff",
       int((~billing_raw.tariff.isin(["Basic", "Comfort", "Premium"])).sum()))
record("3", "дефект: paid_rub с запятой", int(billing_raw.paid_rub.str.contains(",", regex=False).sum()))
record("3", "дефект: пустой traffic_gb", int((billing_raw.traffic_gb == "").sum()))
record("3", "дефект: пустой closing_debt_rub", int((billing_raw.closing_debt_rub == "").sum()))
record("3", "дефект: отрицательные voice_minutes", int((to_num(billing_raw.voice_minutes) < 0).sum()))

print(billing_raw.tariff.value_counts().to_dict())
pd.DataFrame(figures[-7:])

{'Comfort': 9641, 'Basic': 9528, 'Premium': 4879, 'COMFORT': 93, ' basic ': 77, 'Премиум': 30}


,раздел,показатель,значение
0,3,дефект: полные дубли строк,150
1,3,дефект: billing_month в формате ДД.ММ.ГГГГ,400
2,3,дефект: нестандартное написание tariff,200
3,3,дефект: paid_rub с запятой,120
4,3,дефект: пустой traffic_gb,240
5,3,дефект: пустой closing_debt_rub,40
6,3,дефект: отрицательные voice_minutes,80


## 4. Определение оттока

In [13]:
record("4", "строк с нулевой оплатой", int((to_num(billing.paid_rub) == 0).sum()))

2879

## 5. Дата расчёта, горизонт, активная база

In [14]:
# Расторжений первого числа нет: граничное правило «расторжение в день T» на этих данных не срабатывает
term_days = term.dropna().dt.day
assert (term_days != 1).all()

record("5", "день месяца расторжений, минимум", int(term_days.min()))
record("5", "день месяца расторжений, максимум", int(term_days.max()))
record("5", "подключений первого числа месяца", int((act.dt.day == 1).sum()))

67

In [15]:
snapshots = snapshot_table(billing, act, term)
snapshots

,T,N,K,churners,churn_%,no_billing,billing_lt_3m
0,2025-09-01,2000,200,42.0,2.10,2000,2000
1,2025-10-01,1958,195,43.0,2.20,0,1958
2,2025-11-01,1965,196,46.0,2.34,0,1965
3,2025-12-01,1969,196,48.0,2.44,0,95
4,2026-01-01,1971,197,50.0,2.54,0,98
5,2026-02-01,1971,197,52.0,2.64,0,99
6,2026-03-01,1969,196,54.0,2.74,0,99
7,2026-04-01,1965,196,56.0,2.85,0,99
8,2026-05-01,1959,195,58.0,2.96,0,98
9,2026-06-01,1951,195,60.0,3.08,0,98


In [16]:
t = pd.to_datetime(snapshots["T"])
labelled = snapshots.churners.notna()

# Доля оттока растёт каждый месяц
rate = snapshots.loc[labelled, "churn_%"].astype(float)
assert rate.is_monotonic_increasing

# До 01.12.2025 история короче трёх месяцев у всей базы; с 01.12.2025 — у меньшинства (< 10%)
early = t < "2025-12-01"
assert (snapshots.billing_lt_3m[early] == snapshots.N[early]).all()
assert (snapshots.billing_lt_3m[~early & labelled] < 0.1 * snapshots.N[~early & labelled]).all()

train = snapshots[(t >= "2025-12-01") & (t <= "2026-07-01")]
record("5", "доля оттока на 01.09.2025, %", rate.iloc[0])
record("5", "доля оттока на 01.09.2026, %", rate.iloc[-1])
record("5", "обучающая выборка, пар", int(train.N.sum()))
record("5", "обучающая выборка, с оттоком", int(train.churners.sum()))
record("5", "без биллинга на 01.10.2026", int(snapshots.no_billing.iloc[-1]))

50

### Независимая проверка 1: один абонент вручную

C000003 по журналу расторг договор 20.07.2026. Тогда:
- в июле у него 19 активных дней (1–19 июля);
- на 01.07.2026 он в базе и ушёл в этом месяце;
- на 01.08.2026 его в базе уже нет.

Эти строки есть и в `billing_preview.csv` — их можно сверить глазами.

In [17]:
cid = "C000003"
print(events[events.customer_id == cid].to_string(index=False))
print(billing[billing.customer_id == cid].sort_values("bm")[["bm", "active_days", "available_at"]].tail(2).to_string(index=False))

assert term[cid] == pd.Timestamp("2026-07-20")
assert int(billing.loc[(billing.customer_id == cid) & (billing.bm == "2026-07-01"), "active_days"].iloc[0]) == 19
assert cid in active_base(act, term, pd.Timestamp("2026-07-01"))
assert cid not in active_base(act, term, pd.Timestamp("2026-08-01"))

event_id customer_id event_date  event_type
E0000003     C000003 2022-08-18  activation
E0003010     C000003 2026-07-20 termination
        bm active_days        available_at
2026-06-01          30 2026-07-01 00:00:00
2026-07-01          19 2026-08-01 00:00:00


### Независимая проверка 2: активная база только по биллингу

Таблица снимков построена по датам из журнала событий. Здесь та же база считается без журнала — только по наличию строк биллинга:
- абонент активен на T, если у него есть строка за месяц до T (подключён до T) и строка за месяц T (договор ещё действует в месяце T);
- абонент ушёл в месяце T, если строки за следующий месяц нет.

Второе правило верно, потому что расторжений первого числа нет (проверено выше). Если два способа дают одинаковые множества абонентов, правило активной базы реализовано верно.

In [18]:
has_row = set(zip(billing.customer_id, billing.bm))
customers = billing.customer_id.unique()
checked = []
for T in pd.date_range("2025-10-01", "2026-08-01", freq="MS"):
    prev_m, next_m = T - pd.DateOffset(months=1), T + pd.DateOffset(months=1)
    by_billing = {c for c in customers if (c, prev_m) in has_row and (c, T) in has_row}
    by_events = set(active_base(act, term, T))
    assert by_billing == by_events, f"база на {T.date()} не совпала"
    if next_m <= billing.bm.max():
        churn_billing = {c for c in by_billing if (c, next_m) not in has_row}
        churn_events = {c for c in by_events if term[c] < next_m}
        assert churn_billing == churn_events, f"отток на {T.date()} не совпал"
    checked.append(T.date())
print("база совпала на", len(checked), "датах:", checked[0], "–", checked[-1])

база совпала на 11 датах: 2025-10-01 – 2026-08-01


### Пары (абонент, T) для разделов 5 и 7

Снимки 01.12.2025–01.09.2026. Признак долга берётся из строки за прошлый месяц — она доступна на T.

In [19]:
pairs = []
for T in pd.date_range("2025-12-01", "2026-09-01", freq="MS"):
    t_next = T + pd.DateOffset(months=1)
    base = active_base(act, term, T)
    last_month = billing[(billing.bm == T - pd.DateOffset(months=1)) & (billing.available <= T)]
    last_month = last_month.set_index("customer_id").reindex(base)
    pairs.append(pd.DataFrame({
        "T": T,
        "churn": ((term[base] >= T) & (term[base] < t_next)).astype(int).values,
        "tenure_days": (T - act[base]).dt.days.values,
        "debt": to_num(last_month.closing_debt_rub).values,
    }, index=base))
pairs = pd.concat(pairs)
pairs["new"] = pairs.tenure_days < 91
pairs["has_debt"] = pairs.debt > 0
print(len(pairs), "пар")

19541 пар


In [20]:
# Новые абоненты (стаж < 3 месяцев) уходят чаще, но ненамного
by_tenure = pairs.groupby("new").churn.mean() * 100
assert by_tenure[True] > by_tenure[False] and by_tenure[True] / by_tenure[False] < 1.5

record("5", "отток со стажем < 3 мес., %", round(by_tenure[True], 2))
record("5", "отток со стажем ≥ 3 мес., %", round(by_tenure[False], 2))

np.float64(2.87)

## 6. Утечка через строку месяца расторжения

In [21]:
# В месяце расторжения active_days меньше числа дней месяца
days_in_month = (billing.month_end - billing.bm).dt.days
term_month = billing.term < billing.month_end
assert (billing.active_days.astype(int)[term_month] < days_in_month[term_month]).all()

record("6", "строк биллинга в месяце расторжения", int(term_month.sum()))

634

In [22]:
# Что будет, если взять строку месяца T, не глядя на available_at.
# Берём 01.08.2026: строка за август есть в выгрузке, но доступна только с 01.09.
T = pd.Timestamp("2026-08-01")
base = active_base(act, term, T)
row_T = billing[billing.bm == T].set_index("customer_id").reindex(base)
assert row_T.active_days.notna().all(), "у каждого абонента базы есть строка за месяц T"
leak_flag = row_T.active_days.astype(int) < 31
truth = (term[base] >= T) & (term[base] < T + pd.DateOffset(months=1))
print(pd.crosstab(leak_flag.rename("active_days < 31"), truth.rename("ушёл")))

# Правило «active_days меньше числа дней месяца» угадывает отток без ошибок
assert (leak_flag == truth).all()
# Фильтр available_at <= T эту строку отсекает
assert (row_T.available > T).all()

record("6", "утечка на 01.08.2026: угадано уходов без ошибок", int(truth.sum()))

ушёл              False  True 
active_days < 31              
False              1866      0
True                  0     63


63

## 7. Метрики и базовое правило

In [23]:
# Ушедших каждый месяц меньше K, поэтому precision@K ограничена сверху
assert (snapshots.churners[labelled] < snapshots.K[labelled]).all()

test = snapshots[t == "2026-09-01"].iloc[0]
record("7", "тест 01.09.2026: N", int(test.N))
record("7", "тест 01.09.2026: K", int(test.K))
record("7", "тест 01.09.2026: ушли", int(test.churners))
record("7", "предел precision@K на тесте, %", round(test.churners / test.K * 100, 1))

np.float64(34.0)

In [24]:
# Долг на конец прошлого месяца связан с оттоком сильнее всего, что мы проверили
by_debt = pairs.groupby("has_debt").churn.mean() * 100
assert by_debt[True] > 4 * by_debt[False]

record("7", "отток с долгом, %", round(by_debt[True], 2))
record("7", "отток без долга, %", round(by_debt[False], 2))
record("7", "во сколько раз выше с долгом", round(by_debt[True] / by_debt[False], 1))

np.float64(5.2)

In [25]:
# Абонентов с долгом каждый месяц больше K: фильтр не годится, нужна сортировка по сумме долга
per_month = pairs.groupby("T").agg(N=("churn", "size"), with_debt=("has_debt", "sum"), churners=("churn", "sum"))
per_month["K"] = (per_month.N * 0.10).astype(int)
assert (per_month.with_debt > per_month.K).all()

record("7", "абонентов с долгом в месяц, минимум", int(per_month.with_debt.min()))
record("7", "абонентов с долгом в месяц, максимум", int(per_month.with_debt.max()))
per_month

,N,with_debt,churners,K
T,,,,
2025-12-01,1969,455,48,196
2026-01-01,1971,466,50,197
2026-02-01,1971,444,52,197
2026-03-01,1969,448,54,196
2026-04-01,1965,467,56,196
2026-05-01,1959,450,58,195
2026-06-01,1951,492,60,195
2026-07-01,1941,436,62,194
2026-08-01,1929,438,63,192


In [26]:
# Базовое правило на тесте: top-K по сумме долга, равенства по customer_id
test_pairs = pairs[pairs["T"] == "2026-09-01"]
K = int(len(test_pairs) * 0.10)
top = (test_pairs.assign(debt=test_pairs.debt.fillna(0))
       .reset_index(names="customer_id")
       .sort_values(["debt", "customer_id"], ascending=[False, True])
       .head(K))
recall = top.churn.sum() / test_pairs.churn.sum()
precision = top.churn.mean()

# Правило лучше случайного выбора, у которого recall@K ≈ 10%
assert recall > 0.10

record("7", "базовое правило: recall@K на тесте, %", round(recall * 100, 1))
record("7", "базовое правило: precision@K на тесте, %", round(precision * 100, 1))

np.float64(15.7)

## Сводка для устава

Если тетрадь дошла до этой ячейки, все утверждения проверены. Числа ниже — источник для `docs/project_charter.md`; таблица снимков раздела 5 — ячейка `snapshots` выше.

In [27]:
summary = pd.DataFrame(figures)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(summary)
print("Все проверки пройдены")

,раздел,показатель,значение
0,3,строк в subscribers.csv,2600
1,3,строк в billing_monthly.csv,24248
2,3,строк в contract_events.csv,3299
3,3,строк в billing_preview.csv,60
4,3,строк в data_dictionary.csv,23
5,3,подключений,2600
6,3,расторжений,699
7,3,activation_date заполнена,2535
8,3,activation_date пуста,65
9,3,строк биллинга после удаления дублей = ожидаемых,24098


Все проверки пройдены
